In [ ]:
# ============================================
# GOLD LAYER - STAR SCHEMA MODELING
# ============================================

import logging
from pyspark.sql.functions import col, month, year, when

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

# LOAD
sales = spark.read.table("silver_sales_clean")
exchange = spark.read.table("silver_exchange_rate_clean")

# PREPARE
sales = sales.withColumn("month", month(col("order_date"))) \
             .withColumn("year", year(col("order_date")))

df = sales.join(exchange, ["month", "year"], "left")

df = df.withColumn(
    "exchange_rate",
    when(col("exchange_rate").isNull(), 1).otherwise(col("exchange_rate"))
)

df = df.withColumn(
    "revenue_vnd",
    col("final_amount") * col("exchange_rate")
)

# DIMENSIONS
dim_product = sales.select("items").distinct()
dim_product.write.mode("overwrite").saveAsTable("dim_product")

dim_location = sales.select("location").distinct()
dim_location.write.mode("overwrite").saveAsTable("dim_location")

dim_date = sales.select("order_date", "month", "year").distinct()
dim_date.write.mode("overwrite").saveAsTable("dim_date")

# FACT
fact_sales = df.select(
    "order_id",
    "items",
    "location",
    "order_date",
    "revenue_vnd"
)

fact_sales.write.mode("overwrite").saveAsTable("fact_sales")

print("Sample fact table:")
spark.read.table("fact_sales").limit(5).show()

mssparkutils.notebook.exit("Success")